[![../img/pythonista.png](../img/pythonista.png)](https://www.pythonista.io)

# Tamaño del efecto.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo III: *Inferencia estadística*

> 🔑 **Versión con soluciones.** Solo para el instructor o para revisión posterior.

En este *notebook* se reportará el tamaño del efecto (d de Cohen, V de Cramér y η²) junto con el p-value y se integrarán las pruebas de hipótesis vistas en el módulo.

> 📌 **Cómo usar estas soluciones.** Primero intenta cada ejercicio en el *notebook* del alumno (`../22_tamano_del_efecto.ipynb`), sin abrir este archivo; después compara tu código y tu respuesta con los de aquí. Una solución distinta puede ser igual de válida.

**Criterios de revisión.** En cada respuesta que compare grupos o categorías, revisa que estén, cuando apliquen:
1. la **diferencia** en las unidades originales (gramos, milímetros, años, puntos…), con la dirección clara: qué grupo tiene más;
2. su **IC del 95%**, leído como un rango de valores compatibles con los datos (el 95% es del procedimiento, no de este intervalo);
3. el **p-value**, con la decisión de **rechazar** o **no rechazar** H0 (no rechazarla no la demuestra);
4. un **tamaño del efecto** (d de Cohen, V de Cramér, η², riesgo relativo o probabilidad de superioridad) y su magnitud;
5. una **frase sin lenguaje causal**: «se asocia con», «tuvo en promedio», y no «causa», «provoca» ni «hace que».

**Al terminar podrás:**

* elegir el tamaño del efecto que acompaña a cada prueba: d de Cohen, diferencia de proporciones, riesgo relativo, razón de momios, V de Cramér, η² o probabilidad de superioridad;
* leerlo en unidades originales, en desviaciones estándar y en frecuencias naturales;
* acompañarlo con su intervalo *bootstrap*;
* redactar un hallazgo con la plantilla, sin lenguaje causal.

## 🔁 Antes de empezar: repaso de memoria.

Sin abrir `21_proporciones_y_mas_de_dos_grupos.ipynb`, escribe con tus palabras, en tres o cuatro frases, la idea clave de ese *notebook*: cómo se comparan dos proporciones y dos variables de categorías, y qué se hace cuando hay tres o más grupos: qué dice la H1 del ANOVA y qué se hace después. No es un cuestionario ni hay una sola respuesta correcta: se trata de reconstruir la idea de memoria, porque recordar sin mirar ayuda a fijarla. Abajo está un resumen para comparar; léelo solo después de escribir el tuyo.

**✅ Para comparar:**

Dos proporciones se comparan con su diferencia en puntos (con su IC), el riesgo relativo y la prueba *z*; dos variables de categorías, con la chi cuadrada (con frecuencias esperadas de 5 o más) y la V de Cramér. Con tres o más grupos hay una sola H0 (todas las medias son iguales) y una sola H1 (al menos una media difiere): se usa el ANOVA de Welch y, si se rechaza H0, Tukey o Welch por pares con Holm, para no inflar las falsas alarmas. El tamaño del efecto del ANOVA es η², y con muestras enormes hasta una diferencia diminuta sale significativa.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien quiere profundizar. Se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencias.

py311 no cubre inferencia, así que la referencia de la API es la documentación oficial: [*SciPy* · `scipy.stats`](https://docs.scipy.org/doc/scipy/reference/stats.html) (`ttest_ind`, `mannwhitneyu`, `permutation_test`, `chi2_contingency`, `f_oneway`, `levene`), [*statsmodels* · pruebas de proporciones](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.proportions_ztest.html), [*statsmodels* · `pairwise_tukeyhsd`](https://www.statsmodels.org/stable/generated/statsmodels.stats.multicomp.pairwise_tukeyhsd.html) y [*statsmodels* · `anova_oneway`](https://www.statsmodels.org/stable/generated/statsmodels.stats.oneway.anova_oneway.html).

### Preparación del entorno.

* La siguiente celda importará las bibliotecas y las funciones de *SciPy* y *statsmodels* que se usarán, y definirá `DATA` y la semilla `rng`.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.proportion import proportions_ztest

* La siguiente celda leerá `penguins_limpio.csv` (o lo regenerará si no existe), `titanic.csv` y `tips.csv`, y creará `tip_pct`.

In [ ]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio)
titanic = pd.read_csv(os.path.join(DATA, "titanic.csv"))
propinas = pd.read_csv(os.path.join(DATA, "tips.csv")).assign(tip_pct=lambda d: 100 * d["tip"] / d["total_bill"])
pinguinos.shape, titanic.shape, propinas.shape

### Objetos de los *notebooks* anteriores.

Este *notebook* es independiente: la siguiente celda vuelve a crear lo que se necesita de los *notebooks* anteriores.

* La siguiente celda definirá `d_cohen` y `eta_cuadrada`, presentadas en `20_pruebas_para_dos_grupos.ipynb` y `21_proporciones_y_mas_de_dos_grupos.ipynb`.

In [ ]:
def d_cohen(a, b):
    a, b = np.asarray(a), np.asarray(b)
    varianza_combinada = ((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2)
    return (a.mean() - b.mean()) / np.sqrt(varianza_combinada)


def eta_cuadrada(valores, grupos):
    datos = pd.DataFrame({"valor": valores, "grupo": grupos}).dropna()
    media_global = datos["valor"].mean()
    por_grupo = datos.groupby("grupo")["valor"].agg(["count", "mean"])
    suma_entre = (por_grupo["count"] * (por_grupo["mean"] - media_global) ** 2).sum()
    suma_total = ((datos["valor"] - media_global) ** 2).sum()
    return suma_entre / suma_total

---
## 1. Tamaño del efecto y ejercicios de este *notebook*.

| Situación | Prueba | Tamaño del efecto |
|---|---|---|
| Dos medias | Welch | d de Cohen (y la diferencia con su IC, en unidades originales) |
| Dos distribuciones asimétricas | Mann-Whitney o permutación de medianas | Diferencia de medianas con IC *bootstrap* |
| Dos proporciones | *z* de proporciones | Diferencia de proporciones o riesgo relativo |
| Dos categóricas | Chi cuadrada | V de Cramér |
| Tres o más medias | ANOVA de Welch de entrada (Levene solo describe la dispersión) + Tukey | η² |

**Plantilla para redactar:** "En estos datos, [grupo A] tuvo [medida] [mayor/menor] que [grupo B] ([valor A] contra [valor B]; diferencia de [valor y unidades], IC 95% [a, b]; [prueba], *p* [valor]; [tamaño del efecto: d de Cohen, V de Cramér o η²] = [valor], un efecto [magnitud])." Sin verbos causales ("provoca", "hace que") cuando los datos son observacionales.

**¿Por qué hace falta el tamaño del efecto?** Un p-value chico dice que, si en promedio los grupos fueran iguales, una diferencia así de grande, en cualquier dirección, sería rara solo por el azar del muestreo; no dice cuánto difieren ni si importa. Con muestras enormes basta una diferencia diminuta para obtener un p-value chico (en `21_proporciones_y_mas_de_dos_grupos.ipynb` se simula un caso con *p* ≈ 8×10⁻⁶ y d ≈ 0.02): se detecta, pero es demasiado pequeña para que importe en la práctica. El tamaño del efecto contesta la otra pregunta, «¿cuánto?», y se puede dar de **tres maneras**:
* en las **unidades originales**: los machos Adelie pesan unos 675 g más que las hembras;
* en **desviaciones estándar** (la d de Cohen): d = 2.17;
* en **frecuencias naturales**: en unas 94 de cada 100 parejas macho–hembra de Adelie, el macho pesa más.

**Las escalas son orientativas.** Los valores 0.2, 0.5 y 0.8 de la d son convenciones de aproximación: una d de 0.2 puede ser muy importante en salud pública y una de 0.8 irrelevante para otra decisión. Siempre se leen junto con la pregunta que se quiere contestar. Las escalas de V de Cramér (0.1, 0.3 y 0.5, solo para tablas con 2 filas o 2 columnas) y de η² (0.01, 0.06 y 0.14) están en `21_proporciones_y_mas_de_dos_grupos.ipynb`; V no es un porcentaje.

**Un contraste con datos reales.** Los Chinstrap y los Adelie difieren en 32 g de masa media (*p* = 0.59, d = 0.07: un efecto pequeño o nulo; el intervalo de la diferencia, de −86 a 150 g, no descarta diferencias de hasta unos 150 g), y los fumadores y no fumadores de `tips` dejan 16.3% y 15.9% de propina (*p* = 0.68, d = 0.06). En ambos casos no hay evidencia de un efecto, y se ven muy distintos de los 675 g y la d de 2.17 de los Adelie.

> 🤔 **Explícalo con tus palabras.** Si hubiera que contarle la diferencia de masa de los Adelie a alguien que no sabe estadística, ¿cuál de las tres maneras usarías? ¿Y si hubiera que compararla con la diferencia en la longitud de la aleta?

**✅ Una explicación posible:** para alguien sin formación estadística, las frecuencias naturales («en unas 94 de cada 100 parejas, el macho pesa más») o las unidades originales (675 g) se entienden de inmediato. Para comparar con la aleta, que se mide en milímetros, sirve la d, porque está en desviaciones estándar y no depende de las unidades.

**Concepto: tres medidas para dos proporciones.** Cuando se comparan dos tasas (por ejemplo, la supervivencia de mujeres y hombres), el tamaño del efecto se puede dar de tres maneras que **no** son intercambiables:
* **Diferencia de proporciones**, en **puntos**: la tasa de un grupo menos la del otro.
* **Riesgo relativo (RR):** la tasa de un grupo entre la del otro. Un RR de 2 quiere decir «el doble de la tasa». Siempre hay que decir qué grupo va arriba.
* **Razón de momios (OR, por *odds ratio*):** los **momios** de un grupo son sus éxitos entre sus fracasos (por ejemplo, sobrevivientes entre no sobrevivientes); la razón de momios es el cociente de los momios de los dos grupos.

**Advertencia:** la razón de momios **no** es «cuántas veces más probable». Solo se parece al riesgo relativo cuando las tasas son chicas (digamos, menores que 10%). Con tasas grandes, la OR exagera mucho la diferencia.

**Ejemplo:** si 8 de cada 10 personas de un grupo y 2 de cada 10 del otro tienen el resultado, la diferencia es de 60 puntos y el RR es 4 (80% entre 20%). Los momios son 8/2 = 4 y 2/8 = 0.25, así que la OR es 16: cuatro veces el RR.

* La siguiente celda calculará las tres medidas para el sexo y la supervivencia en el Titanic, junto con la chi cuadrada sin corrección y la V de Cramér.

In [ ]:
tabla_sexo = pd.crosstab(titanic["sex"], titanic["survived"])
tasa_sexo = tabla_sexo[1] / tabla_sexo.sum(axis=1)
momios_sexo = tabla_sexo[1] / tabla_sexo[0]
chi2_sexo = stats.chi2_contingency(tabla_sexo, correction=False)
{"tasa mujeres": round(float(tasa_sexo["female"]), 3),
 "tasa hombres": round(float(tasa_sexo["male"]), 3),
 "diferencia (puntos)": round(float(100 * (tasa_sexo["female"] - tasa_sexo["male"])), 1),
 "riesgo relativo": round(float(tasa_sexo["female"] / tasa_sexo["male"]), 2),
 "momios mujeres": round(float(momios_sexo["female"]), 2),
 "momios hombres": round(float(momios_sexo["male"]), 3),
 "razón de momios": round(float(momios_sexo["female"] / momios_sexo["male"]), 1),
 "chi cuadrada": round(float(chi2_sexo.statistic), 1),
 "p-value": float(chi2_sexo.pvalue),
 "V de Cramér": round(float(stats.contingency.association(tabla_sexo, method="cramer")), 2)}

**Cómo leer el resultado.**

* **Diferencia de proporciones:** sobrevivió 74.2% de las mujeres y 18.9% de los hombres, una diferencia de 55.3 **puntos** (no «55.3%»): de cada 100 pasajeras sobrevivieron unas 55 personas más que de cada 100 pasajeros.
* **Riesgo relativo:** 3.93. La tasa de supervivencia de las mujeres fue unas 3.9 veces la de los hombres. Al revés (hombres entre mujeres), sale 0.25.
* **Razón de momios:** los momios fueron 2.88 en mujeres (unas 288 sobrevivientes por cada 100 que no sobrevivieron) y 0.233 en hombres, así que la razón de momios es 12.4. **No** quiere decir «12 veces más probable»: las tasas no son chicas, y la OR queda muy por encima del RR.
* **p-value y V de Cramér:** con χ² = 263.1, *p* < 0.001: si en la población el sexo y la supervivencia no estuvieran asociados, una tabla tan desequilibrada casi nunca saldría por azar. La V de Cramér es 0.54. Como la tabla tiene una variable de sí o no, aplican las referencias orientativas 0.1, 0.3 y 0.5: la asociación es grande. Con tablas más grandes las referencias bajan (en una tabla de 3 por 3 son unos 0.07, 0.21 y 0.35).
* **Alcance:** son datos observacionales de un solo barco. El sexo coincide con otros factores (la clase, la ubicación de los camarotes, las reglas para subir a los botes), y estos datos no permiten separar su papel.

> 🤔 **Explícalo con tus palabras.** ¿Por qué la razón de momios (12.4) sale tanto mayor que el riesgo relativo (3.9)? ¿Cuál de las dos usarías para contarle el resultado a alguien sin formación estadística?

**✅ Una explicación posible:** los momios dividen entre los que **no** sobrevivieron. En las mujeres, que sobrevivieron mucho, ese denominador es chico (25.8%) y los momios se disparan a 2.88; en los hombres casi no cambian respecto a la tasa (0.233 contra 0.189). Por eso la OR multiplica el efecto. Para comunicar sirven mejor la diferencia en puntos o el riesgo relativo, que se leen como tasas.

**¿Por qué esta sección?** El producto final de una prueba no es el p-value, sino una **frase** que una persona sin formación estadística pueda leer y que no exagere. La tabla resume qué tamaño del efecto acompaña a cada prueba, y la plantilla sirve para redactar los hallazgos de un informe (como en `25_estructura_del_informe_y_comunicacion.ipynb` y `26_del_notebook_al_informe.ipynb`).

**💡 Ideas clave de la sección 1:** cada prueba tiene su tamaño del efecto; se reporta magnitud, IC, prueba y p-value, en lenguaje descriptivo. Para dos proporciones, la diferencia en puntos, el riesgo relativo y la razón de momios no son intercambiables: la razón de momios solo se parece al riesgo relativo cuando las tasas son chicas.

### Ejercicio 1a · Edad y supervivencia en el Titanic.

**🎯 Objetivo:** calcular una comparación de dos medias y redactarla con la plantilla. **Qué se espera aprender:** a comunicar magnitud, incertidumbre, prueba y tamaño del efecto en una sola frase, sin lenguaje causal, también cuando el p-value es chico pero el efecto es pequeño.

La masa de machos y hembras Adelie ya sirvió de ejemplo en la sección 1. Aquí se usará un caso nuevo: la **edad** (`age`, en años) de los pasajeros del Titanic que sobrevivieron y la de los que no sobrevivieron. 177 pasajeros no tienen edad registrada; quedan 290 que sobrevivieron y 424 que no, suficientes para que Welch funcione bien. Los datos son observados: describen a estos pasajeros, no una causa.

Sigue estos pasos con el *DataFrame* `titanic`:

1. **Hipótesis.** Escribe en palabras H0 y H1 (prueba bilateral).
2. **Prepara y describe.** Quita los faltantes de `age`, separa la edad de quienes sobrevivieron (`survived == 1`) y de quienes no (`survived == 0`) y calcula *n* y media de cada grupo.
3. **Welch e IC.** Aplica `stats.ttest_ind(..., equal_var=False)` y obtén el IC del 95% de la diferencia de medias (sobrevivieron menos no sobrevivieron). Con α = 0.05, decide si se **rechaza** o **no se rechaza** H0.
4. **Tamaño del efecto.** Calcula la d de Cohen con `d_cohen`.
5. **Redacción.** Con la plantilla de la sección 1, redacta la conclusión en una o dos frases. Di también si una diferencia de ese tamaño te parece importante y por qué.

* La siguiente celda muestra una solución posible.

In [ ]:
con_edad = titanic.dropna(subset=["age"])
edad_sobrevivio = con_edad.loc[con_edad["survived"] == 1, "age"].to_numpy()
edad_no_sobrevivio = con_edad.loc[con_edad["survived"] == 0, "age"].to_numpy()
prueba_edad = stats.ttest_ind(edad_sobrevivio, edad_no_sobrevivio, equal_var=False)
ic_edad = prueba_edad.confidence_interval(confidence_level=0.95)
{"n": (len(edad_sobrevivio), len(edad_no_sobrevivio)),
 "medias (años)": (round(float(edad_sobrevivio.mean()), 1), round(float(edad_no_sobrevivio.mean()), 1)),
 "diferencia (años)": round(float(edad_sobrevivio.mean() - edad_no_sobrevivio.mean()), 2),
 "IC 95% (años)": (round(float(ic_edad.low), 2), round(float(ic_edad.high), 2)),
 "p-value": round(float(prueba_edad.pvalue), 3),
 "d de Cohen": round(float(d_cohen(edad_sobrevivio, edad_no_sobrevivio)), 2)}

**✅ Respuesta sugerida:**

**H0:** en la población de pasajeros, la edad media de quienes sobrevivieron es igual a la de quienes no sobrevivieron. **H1:** las dos edades medias son distintas.

En estos datos, los pasajeros que sobrevivieron tenían en promedio 2.3 años menos que los que no sobrevivieron (28.3 contra 30.6 años; diferencia de −2.3 años, IC 95% −4.5 a −0.1 años; *t* de Welch, p-value = 0.04; d de Cohen = −0.16, un efecto pequeño según la referencia de Cohen, que es orientativa).

Con α = 0.05 se rechaza H0, pero el efecto es pequeño y el IC llega casi hasta el 0: los datos son compatibles con una diferencia de entre unos pocos meses y unos 4.5 años. Además, a 177 pasajeros les falta la edad; si esos datos no faltan al azar, la cifra podría cambiar. Es una asociación observacional: la edad viene junto con el sexo, la clase y las reglas para subir a los botes, y estos datos no permiten decir que ser más joven ayudara a sobrevivir.

### Ejercicio 1b · Aleta por especie.

**🎯 Objetivo:** recorrer el flujo completo para más de dos grupos (ANOVA de Welch → Tukey → η²), con Levene solo como descripción de la dispersión. **Qué se espera aprender:** a usar Welch por defecto, sin que Levene decida qué prueba usar, y a redactar la conclusión con el tamaño del efecto.

Compara `flipper_length_mm` entre las tres especies: calcula Levene solo para describir si la dispersión es parecida entre especies, aplica el ANOVA de Welch (y, para comparar, el ANOVA clásico), calcula η² y, si el ANOVA de Welch rechaza H0, Tukey. Redacta la conclusión.

* La siguiente celda muestra una solución posible.

In [ ]:
con_aleta = pinguinos.dropna(subset=["flipper_length_mm"])
grupos_aleta = [grupo["flipper_length_mm"].to_numpy() for _, grupo in con_aleta.groupby("species")]
resumen_anova = {
    "Levene p": round(float(stats.levene(*grupos_aleta).pvalue), 3),
    "ANOVA F": round(float(stats.f_oneway(*grupos_aleta).statistic), 1),
    "ANOVA p": float(stats.f_oneway(*grupos_aleta).pvalue),
    "Welch p": float(anova_oneway(con_aleta["flipper_length_mm"], con_aleta["species"], use_var="unequal").pvalue),
    "eta²": round(float(eta_cuadrada(con_aleta["flipper_length_mm"], con_aleta["species"])), 3),
}
tukey = pairwise_tukeyhsd(con_aleta["flipper_length_mm"], con_aleta["species"])
tabla_tukey = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])
resumen_anova, tabla_tukey

**✅ Respuesta sugerida:**

El ANOVA de Welch, que se usa de entrada, es contundente (*p* < 0.001) y el ANOVA clásico llega a la misma conclusión (F = 594.8, *p* < 0.001). Levene (*p* = 0.72) describe dispersiones parecidas entre especies; es una descripción, no cambia la prueba elegida, aunque sí respalda leer los intervalos de Tukey sin reservas. **Conclusión:** en estos datos el largo de la aleta difiere entre las tres especies (medias de 190, 196 y 217 mm); Tukey confirma que difieren todos los pares (Gentoo supera a Adelie en ≈ 27 mm y a Chinstrap en ≈ 21 mm; Chinstrap supera a Adelie en ≈ 6 mm) y la especie se asocia con cerca de 78% de la variabilidad de la aleta (η² = 0.78, un efecto muy grande).

### Ejercicio 1c · Puerto de embarque y supervivencia.

**🎯 Objetivo:** cuantificar una asociación entre categóricas. **Qué se espera aprender:** a acompañar la chi cuadrada con la V de Cramér y las tasas por grupo.

La clase del boleto y la supervivencia ya se analizaron en `21_proporciones_y_mas_de_dos_grupos.ipynb`. Aquí se usará otra variable del Titanic: el puerto donde embarcó cada pasajero (`embark_town`: Cherbourg, Queenstown o Southampton). Dos pasajeros no tienen puerto registrado; quedan 889, con al menos 77 por puerto.

Sigue estos pasos con el *DataFrame* `titanic`:

1. **Hipótesis.** Escribe en palabras H0 (el puerto y la supervivencia no están asociados) y H1.
2. **Tabla y tasas.** Arma la tabla con `pd.crosstab(titanic["embark_town"], titanic["survived"])` (`crosstab` deja fuera los faltantes) y calcula la tasa de supervivencia de cada puerto.
3. **Chi cuadrada.** Aplica `stats.chi2_contingency(tabla)` y reporta el estadístico, los grados de libertad, el p-value y la menor frecuencia esperada (debe ser de 5 o más).
4. **V de Cramér.** Calcúlala con `stats.contingency.association(tabla, method="cramer")`. La tabla tiene 2 columnas, así que aplican las referencias orientativas 0.1, 0.3 y 0.5.
5. **Conclusión.** Redáctala sin lenguaje causal. ¿Qué otras variables podrían venir junto con el puerto?

* La siguiente celda muestra una solución posible.

In [ ]:
tabla_puerto = pd.crosstab(titanic["embark_town"], titanic["survived"])
chi2_puerto = stats.chi2_contingency(tabla_puerto)
tasas_puerto = (tabla_puerto[1] / tabla_puerto.sum(axis=1)).round(3)
v_puerto = stats.contingency.association(tabla_puerto, method="cramer")
(tabla_puerto, tasas_puerto,
 {"chi cuadrada": round(float(chi2_puerto.statistic), 1),
  "grados de libertad": int(chi2_puerto.dof),
  "p-value": float(chi2_puerto.pvalue),
  "menor frecuencia esperada": round(float(chi2_puerto.expected_freq.min()), 1),
  "V de Cramér": round(float(v_puerto), 2)})

**✅ Respuesta sugerida:**

**H0:** en la población de pasajeros, el puerto de embarque y la supervivencia no están asociados. **H1:** sí están asociados.

La supervivencia se asoció con el puerto de embarque (χ² = 26.5, 2 g. l., p-value < 0.001; V de Cramér = 0.17, una asociación pequeña): sobrevivió 55% de quienes embarcaron en Cherbourg, 39% en Queenstown y 34% en Southampton. La menor frecuencia esperada es de 29.4, así que la chi cuadrada es confiable. Es una asociación observacional y débil: el puerto coincide con otros factores, sobre todo la clase del boleto (cerca de la mitad de quienes embarcaron en Cherbourg viajaba en primera clase, y casi todos los de Queenstown, en tercera), así que estos datos no permiten separar el papel del puerto.

### Ejercicio 2 · Probabilidad de superioridad.

**🎯 Objetivo:** leer el tamaño del efecto en frecuencias naturales. **Qué se espera aprender:** a calcular la probabilidad de superioridad a partir de Mann-Whitney y a compararla con la d de Cohen.

La **probabilidad de superioridad** contesta: si se toma al azar un caso de cada grupo, ¿qué tan seguido el del primer grupo tiene el valor mayor? (los empates cuentan la mitad). Se obtiene al dividir el estadístico U de `stats.mannwhitneyu(a, b)` entre el número de parejas posibles, `len(a) * len(b)`. Un valor de 0.5 significa que ningún grupo tiende a ganar.

Se usarán los tres *DataFrames* de este *notebook*. `titanic` (archivo `titanic.csv`) registra 891 pasajeros del Titanic: la clase, el sexo, la edad, la tarifa pagada y si sobrevivieron. `pinguinos` (archivo `penguins_limpio.csv`) registra 342 pingüinos de tres especies (Adelie, Chinstrap y Gentoo): la isla, las medidas del pico y de la aleta, la masa y el sexo (registrado en 333 de ellos). `propinas` (archivo `tips.csv`) registra 244 mesas de un restaurante: la cuenta, la propina, el día, el horario, el tamaño del grupo y si en la mesa había fumadores; `tip_pct` es la propina como porcentaje de la cuenta. De ellos salen tres comparaciones distintas de las de la sección 1, todas con al menos 34 casos por grupo:

1. la tarifa (`fare`) de primera clase contra la de segunda clase del Titanic (216 y 184 pasajeros);
2. la longitud del pico (`bill_length_mm`) de machos contra hembras Chinstrap (34 y 34);
3. `tip_pct` de mujeres contra hombres en `tips` (`sex` es el sexo de quien pagó; 87 y 157 mesas).

Para cada una, calcula la d de Cohen, la probabilidad de superioridad y las dos medianas. ¿Cómo se lee cada probabilidad en «parejas de cada 100»? ¿En cuál de las tres la d y la probabilidad de superioridad cuentan historias algo distintas, y por qué?

* La siguiente celda muestra una solución posible.

In [ ]:
def prob_superioridad(a, b):
    return stats.mannwhitneyu(a, b).statistic / (len(a) * len(b))


tarifa_primera = titanic.loc[titanic["class"] == "First", "fare"].to_numpy()
tarifa_segunda = titanic.loc[titanic["class"] == "Second", "fare"].to_numpy()
chinstrap = pinguinos[pinguinos["species"] == "Chinstrap"]
pico_machos_chinstrap = chinstrap.loc[chinstrap["sex"] == "Male", "bill_length_mm"].to_numpy()
pico_hembras_chinstrap = chinstrap.loc[chinstrap["sex"] == "Female", "bill_length_mm"].to_numpy()
propina_mujeres = propinas.loc[propinas["sex"] == "Female", "tip_pct"].to_numpy()
propina_hombres = propinas.loc[propinas["sex"] == "Male", "tip_pct"].to_numpy()
comparaciones = {
    "tarifa: primera vs. segunda clase": (tarifa_primera, tarifa_segunda),
    "pico: machos vs. hembras Chinstrap": (pico_machos_chinstrap, pico_hembras_chinstrap),
    "propina: mujeres vs. hombres": (propina_mujeres, propina_hombres),
}
pd.DataFrame({
    "n": [f"{len(a)} y {len(b)}" for a, b in comparaciones.values()],
    "medianas": [f"{np.median(a):.2f} y {np.median(b):.2f}" for a, b in comparaciones.values()],
    "d de Cohen": [d_cohen(a, b) for a, b in comparaciones.values()],
    "probabilidad de superioridad": [prob_superioridad(a, b) for a, b in comparaciones.values()],
}, index=list(comparaciones)).round(2)

**✅ Respuesta sugerida:**

* **Tarifa de primera contra segunda clase:** d = 1.09 y probabilidad de superioridad de 0.92: si se toman al azar un pasajero de primera y uno de segunda, en unas 92 de cada 100 parejas el de primera pagó más (medianas de 60.29 y 14.25). Aquí las dos medidas cuentan historias algo distintas: la tarifa es muy asimétrica y en primera clase hay boletos carísimos, que inflan la desviación estándar y hacen que la d parezca más modesta de lo que se ve al comparar parejas. Si los datos tuvieran forma de campana, una d de 1.09 correspondería a una probabilidad de cerca de 0.78, no de 0.92.
* **Pico de machos contra hembras Chinstrap:** d = 1.84 y 0.92: en unas 92 de cada 100 parejas macho–hembra, el macho tiene el pico más largo. Con datos casi simétricos, las dos medidas cuentan lo mismo. Ojo: no quiere decir que 92% de los machos tenga el pico más largo que todas las hembras.
* **Propina porcentual de mujeres contra hombres:** d = 0.14 y 0.56: en unas 56 de cada 100 parejas la mujer dejó un porcentaje mayor, apenas más que lanzar una moneda. Las dos medidas quedan cerca de «sin efecto» (0 para la d y 0.5 para la probabilidad).

La probabilidad de superioridad no supone que los datos tengan forma de campana y se entiende sin saber estadística, por eso sirve para comunicar, sobre todo con datos asimétricos como la tarifa. Como siempre, son datos observados: describen una asociación, no una causa.

### Ejercicio 3 · Viajar solo y supervivencia en el Titanic.

**🎯 Objetivo:** aplicar a un caso nuevo las tres medidas para dos proporciones de la sección 1. **Qué se espera aprender:** a calcular e interpretar la diferencia de proporciones, el riesgo relativo y la razón de momios, y a leer la V de Cramér con la escala que corresponde al tamaño de la tabla.

El sexo y la supervivencia ya se analizaron en la sección 1. Aquí se usará otra variable: `alone`, que vale `True` si el pasajero viajaba sin familiares a bordo (537 pasajeros) y `False` si viajaba acompañado (354). Con `pd.crosstab(titanic["alone"], titanic["survived"])` calcula:
* la tasa de supervivencia de quienes viajaban acompañados y de quienes viajaban solos, y su **diferencia en puntos** (acompañados menos solos);
* el **riesgo relativo** (tasa de los acompañados entre tasa de los solos);
* la **razón de momios** (acompañados entre solos), como en el ejemplo del sexo de la sección 1;
* la chi cuadrada sin corrección (`stats.chi2_contingency(tabla, correction=False)`) con su p-value, y la **V de Cramér** de viajar solo. Compárala con la del puerto del ejercicio 1c.

Redacta la conclusión sin lenguaje causal.

* La siguiente celda muestra una solución posible.

In [ ]:
tabla_solo = pd.crosstab(titanic["alone"], titanic["survived"])
tasa_solo = tabla_solo[1] / tabla_solo.sum(axis=1)
momios_solo = tabla_solo[1] / tabla_solo[0]
chi2_solo = stats.chi2_contingency(tabla_solo, correction=False)
{"tasa acompañados": round(float(tasa_solo[False]), 3),
 "tasa solos": round(float(tasa_solo[True]), 3),
 "diferencia (puntos)": round(float(100 * (tasa_solo[False] - tasa_solo[True])), 1),
 "riesgo relativo": round(float(tasa_solo[False] / tasa_solo[True]), 2),
 "momios acompañados": round(float(momios_solo[False]), 2),
 "momios solos": round(float(momios_solo[True]), 3),
 "razón de momios": round(float(momios_solo[False] / momios_solo[True]), 2),
 "chi cuadrada": round(float(chi2_solo.statistic), 1),
 "p-value": float(chi2_solo.pvalue),
 "V de Cramér viajar solo": round(float(stats.contingency.association(tabla_solo, method="cramer")), 2),
 "V de Cramér puerto": round(float(v_puerto), 2)}

**✅ Respuesta sugerida:**

* **Diferencia de proporciones:** sobrevivió 50.6% de quienes viajaban acompañados y 30.4% de quienes viajaban solos, una diferencia de 20.2 **puntos** (no «20.2%»): de cada 100 pasajeros acompañados sobrevivieron unos 20 más que de cada 100 que viajaban solos.
* **Riesgo relativo:** 1.67. La tasa de supervivencia de quienes viajaban acompañados fue unas 1.7 veces la de quienes viajaban solos. Siempre hay que decir qué grupo va arriba: al revés, sale 0.60.
* **Razón de momios:** los momios fueron 1.02 entre los acompañados (unos 102 sobrevivientes por cada 100 que no sobrevivieron) y 0.436 entre los solos, así que la razón de momios es 2.35. **No** quiere decir «2.35 veces más probable»: solo se parece al riesgo relativo cuando las tasas son chicas, y aquí no lo son (por eso 2.35 queda lejos de 1.67).
* **p-value y V de Cramér:** con χ² = 36.9, p-value < 0.001. Si en la población viajar solo y sobrevivir no estuvieran asociados, una tabla tan desequilibrada como esta casi nunca saldría solo por el azar del muestreo, así que se rechaza la hipótesis de no asociación. La V de Cramér es de 0.20 para viajar solo y de 0.17 para el puerto: las dos tablas tienen una variable de sí o no, así que aplican las referencias orientativas 0.1, 0.3 y 0.5, y las dos asociaciones son pequeñas, la de viajar solo un poco mayor.

**Conclusión:** en estos datos, viajar acompañado se asoció con una supervivencia más alta. Son datos observacionales de un solo barco: quienes viajaban solos eran sobre todo hombres adultos (76%), mientras que entre los acompañados había muchas mujeres y niños, así que el sexo y la edad vienen junto con esta variable y estos datos no permiten separar su papel.

### Reto 1 · Diagnóstico gráfico de supuestos.

**🎯 Objetivo:** revisar los supuestos del ANOVA con gráficas. **Qué se espera aprender:** a juzgar si una desviación de los supuestos puede cambiar la conclusión.

Para el ejercicio 1b, dibuja un gráfico Q-Q de `flipper_length_mm` de cada especie (`stats.probplot(..., plot=ax)`) y un diagrama de caja por especie. ¿Las conclusiones del ANOVA dependen de algún supuesto dudoso?

* La siguiente celda muestra una solución posible.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (especie, grupo) in zip(axes, con_aleta.groupby("species")):
    stats.probplot(grupo["flipper_length_mm"], plot=ax)
    ax.set_title(f"Q-Q {especie}")
sns.boxplot(con_aleta, x="species", y="flipper_length_mm", ax=axes[3])
axes[3].set(xlabel="especie", ylabel="aleta (mm)", title="Aleta por especie")
plt.tight_layout()
plt.show()

**✅ Respuesta sugerida:**

Los puntos siguen la recta en las tres especies, con escalones porque la aleta se mide en milímetros enteros, y las cajas tienen dispersiones parecidas (Levene no rechaza). Con efectos tan grandes (η² ≈ 0.78) y *n* de 68 a 151 por grupo, ninguna desviación menor cambiaría la conclusión.

### Reto 2 · Intervalo *bootstrap* de la d de Cohen.

**🎯 Objetivo:** acompañar un tamaño del efecto con su incertidumbre. **Qué se espera aprender:** que la d también cambia de muestra a muestra y que su intervalo se puede obtener por *bootstrap*, como en `18_bootstrap_e_interpretacion_de_intervalos.ipynb`.

Con los arreglos de edades del ejercicio 1a (si usaste otros nombres, cámbialos), calcula el IC 95% de la d de Cohen con `stats.bootstrap((edad_sobrevivio, edad_no_sobrevivio), d_cohen, n_resamples=9999, method="percentile", random_state=401, vectorized=False)`. ¿Qué dice el intervalo? ¿El efecto podría ser prácticamente nulo? ¿Podría ser mediano?

* La siguiente celda muestra una solución posible.

In [ ]:
ic_d = stats.bootstrap((edad_sobrevivio, edad_no_sobrevivio), d_cohen, n_resamples=9999, method="percentile",
                       random_state=401, vectorized=False).confidence_interval
{"d de Cohen": round(float(d_cohen(edad_sobrevivio, edad_no_sobrevivio)), 2),
 "IC 95% bootstrap": (round(float(ic_d.low), 2), round(float(ic_d.high), 2))}

**✅ Respuesta sugerida:**

La d es de −0.16 y su intervalo *bootstrap* va de −0.31 a −0.01. El 95% describe el método: si se repitiera el muestreo muchas veces, los intervalos construidos así cubrirían la d real en cerca de 95% de los casos. Este intervalo en particular no da certeza, y supone que los pasajeros con edad registrada se parecen a una muestra al azar.

El intervalo llega casi hasta el 0, así que los datos son compatibles con un efecto prácticamente nulo; su otro extremo, −0.31, sigue lejos de un efecto mediano (0.5). Lo que se puede decir es que la diferencia de edad entre quienes sobrevivieron y quienes no fue, a lo más, pequeña. Con otra semilla o con otro número de remuestreos, el segundo decimal cambia un poco: que el resultado sea reproducible gracias a la semilla no lo hace más preciso.

---
## ✅ Cierre.

**En este *notebook*:**
* El tamaño del efecto contesta «¿cuánto?» y se puede dar en unidades originales, en desviaciones estándar (d de Cohen) o en frecuencias naturales (probabilidad de superioridad).
* Para categóricas: la diferencia de proporciones (en puntos), el riesgo relativo, la razón de momios y la V de Cramér, cuyas referencias dependen del tamaño de la tabla.
* Para tres o más grupos: η².
* El tamaño del efecto también tiene incertidumbre, que se puede estimar con un intervalo *bootstrap*.
* Las referencias de Cohen son orientativas, y con *n* enorme significativo no implica importante.

**Siguiente *notebook*:** `23_eda_completo_con_mpg.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>